In [1]:
from datetime import datetime
import glob
import hisepy as hp
import logging
import os
import session_info
import shutil
import subprocess

In [2]:
in_uuid = ['b5f6de81-5ee3-4ed7-b737-e273065ec23c']
hp.cache_files(in_uuid)

2026-09-03 12:12:00,765 INFO [hisepy.logging:185] logging 3408 132840475895616 Calling cache_files
2026-09-03 12:12:06,976 INFO [hisepy.logging:228] logging 3408 132840475895616 Finished cache_files successfully (time_elapsed=3.173s)


['/home/workspace/input/1918706177/cohorts/b5f6de81-5ee3-4ed7-b737-e273065ec23c/californium-manganese-carbon/tcell-vrd_deg_vis_2026-07-29.tar']

In [3]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

In [4]:
def _get_git_info() -> str:
    """ Fetch current git branch and commit hash for deployment reproducibility """
    commit_short_hash = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode('ascii').strip()
    branch_name = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode('utf-8').strip()
    return f'{branch_name}_{commit_short_hash}'

In [5]:
def _get_all_file_paths(directory: str) -> list[str]:
    """
    Walk through a directory and get files for app to pass to save_dash_app().
    """
    exclude_exts = ('.ipynb', '.pyc')
    file_paths = [
        os.path.abspath(os.path.join(root, f))
        for root, _, files in os.walk(directory)
        if 'zarrs' not in root
        for f in files
        if not f.endswith('app.py') and not f.endswith(exclude_exts)
    ]
    return file_paths

In [6]:
def deploy_app(app_config: dict, in_uuids: list, ts: str, git_info: str):

    app_title = app_config['app_title']
    app_description = f"{app_config['app_description']} git_info: {git_info}"
      
    study_space_id = app_config['study_space_id']
    project_dir = app_config['project_dir']
    app_dir = f'/home/workspace/{project_dir}_app'
    cwd = os.getcwd()

    shutil.copytree(cwd, app_dir, dirs_exist_ok=True)
      
    all_files = _get_all_file_paths(app_dir)

    app_file = f'{app_dir}/{app_config['app_file']}'
    app_dirs = [f'{app_dir}/{d}' for d in app_config['app_dirs']]

    log_msg = f"""
    === Deploying Dash App: {app_title} ===
    app_filepath         : {app_file}
    additional_files     : {len(all_files)} files included
    input_file_ids       : {in_uuids}
    study_space_id       : {study_space_id}
    title                : {app_title} DEG Explorer (v{app_config['version']}) {ts}
    description          : {app_description}
    image                : {app_dir}/hero_image.png
    requirements         : {app_dir}/pixi.toml
    additional_dirs      : {app_dirs}
    ======================================="""
    logger.info(log_msg) 

    response = hp.save_visualization_app(
        application_files=all_files,
        application_dirs=[],
        study_space_id=study_space_id,
        title=f"{app_title} DEG Explorer (v{app_config['version']}, git:{git_info}) {ts}",
        png_image= f'{app_dir}/hero_image.png',
        data_mount_path = 'data_mount/',
        data_source_file_ids=in_uuids,
        description=app_description,
        build_template_name = 'dash',
        build_template_major_version = 3,
        build_template_minor_version = 4,
        infer_build_template_arguments = True
    )

    logger.info(response)
    return response

In [7]:
app_config = {
    'app_title': 'VRd T cell',
    'version': '2.0.0',
    'app_description': 'Visualization of DEGs from VRd treatment of T cells',
    'project_dir': 'okada_vrd_deg',
    'study_space_id': '40df6403-29f0-4b45-ab7d-f46d420c422e',
    'app_file': 'app.py',
    'app_dirs': ['assets','data','src']
}
git_info = _get_git_info()
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

In [8]:
session_info.show()

In [9]:
deploy_app(
    app_config = app_config, 
    in_uuids = ['b5f6de81-5ee3-4ed7-b737-e273065ec23c'],
    ts = ts,
    git_info = git_info
)

2026-09-03 12:12:07,186 INFO [__main__:30] 2276133927 3408 132840475895616 
    === Deploying Dash App: VRd T cell ===
    app_filepath         : /home/workspace/okada_vrd_deg_app/app.py
    additional_files     : 45 files included
    input_file_ids       : ['b5f6de81-5ee3-4ed7-b737-e273065ec23c']
    study_space_id       : 40df6403-29f0-4b45-ab7d-f46d420c422e
    title                : VRd T cell DEG Explorer (v2.0.0) 20260903_121207
    description          : Visualization of DEGs from VRd treatment of T cells git_info: development_18f6f63
    image                : /home/workspace/okada_vrd_deg_app/hero_image.png
    requirements         : /home/workspace/okada_vrd_deg_app/pixi.toml
    additional_dirs      : ['/home/workspace/okada_vrd_deg_app/assets', '/home/workspace/okada_vrd_deg_app/data', '/home/workspace/okada_vrd_deg_app/src']
2026-09-03 12:12:07,187 INFO [hisepy.logging:185] logging 3408 132840475895616 Calling save_visualization_app


Please enter the path to your app's main Python file: /home/workspace/okada_vrd_deg_app/app.py


2026-09-03 12:12:21,383 INFO [hisepy.logging:366] upload 3408 132840475895616 Created temporary directory for Visualization App build: /home/workspace/temp/tmpx96ozjp1
2026-09-03 12:12:21,943 INFO [hisepy.logging:185] logging 3408 132840475895616 Calling save_static_image
2026-09-03 12:12:28,070 INFO [hisepy.logging:228] logging 3408 132840475895616 Finished save_static_image successfully (time_elapsed=2.994s)
2026-09-03 12:12:28,070 INFO [hisepy.logging:385] upload 3408 132840475895616 Creating Visualization App workflow: https://allenimmunology.org/ide-nextgen/visualization/viz-app/workflow
2026-09-03 12:12:33,249 INFO [hisepy.logging:228] logging 3408 132840475895616 Finished save_visualization_app successfully (time_elapsed=23.151s)
2026-09-03 12:12:33,250 INFO [__main__:47] 2276133927 3408 132840475895616 Visualization App Workflow initiated: https://allenimmunology.org/workflow/flow/602fa4d8-47c3-4c93-8ecc-2939176be5c4


'Visualization App Workflow initiated: https://allenimmunology.org/workflow/flow/602fa4d8-47c3-4c93-8ecc-2939176be5c4'